In [50]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import Field,BaseModel
import pandas as pd 

In [51]:
llm = ChatOpenAI(model='gpt-5-mini')

In [52]:
file_name = input('Enter the file name:')
df = pd.read_csv(f'Data/{file_name}.csv')
schema = df.dtypes.to_string


In [53]:
Prompts = ChatPromptTemplate.from_messages([
    
        ('system',"""
                You are a senior Data Engineer and Data Quality Engineer.

                Analyze the provided dataset information and identify potential data quality issues.

                Check for:
                1. Missing values
                2. Duplicate records
                3. Invalid values
                4. Incorrect data types
                5. Invalid formats
                6. Outliers or suspicious values
                7. Inconsistent values
                8. Potential business-rule violations

                For every issue:
                - Explain the problem
                - Identify the affected column
                - Explain why it is a problem
                - Give a practical recommendation to fix it

                At the end, provide an overall data quality score out of 100.

                Be concise and explain the findings in a way that a Data Engineer can easily understand.
        """),
        ('human',
         """ 
                Analyze the following dataset:

                Dataset name:
                {dataset_name}

                Schema:
                {schema}

                Sample records:
                {sample_data}

                Provide a complete data quality analysis.
         """
        )
])

In [54]:
class the_structure(BaseModel):
    missing: str = Field(description='List the columns that contain missing or NULL values. Mention the column names and briefly describe the missing-value issue.')
    Invalid: str = Field(description='Identify invalid or incorrect values in the dataset, such as negative ages, impossible values, invalid formats, or values that violate expected data types or rules.')
    Potential_issues:str = Field(description='Mention possible data quality problems that may not be immediately invalid, such as duplicates, inconsistent values, unusual patterns, outliers, or possible business-rule violations')
    Score:int = Field(description='Give an overall data quality score from 0 to 100 based on the severity and number of data quality issues. Briefly explain why the score was given.')

In [55]:
struc_llm = llm.with_structured_output(the_structure)

In [56]:
chain = Prompts | struc_llm

In [57]:
response=chain.invoke(
        {
            'dataset_name':file_name,
            'schema':schema,
            'sample_data':df
        }
)

In [70]:
for key, value in response.model_dump().items():
    print(f"{key}:")
    print(value)
    print()

missing:
Columns with missing/NULL values and brief description:
- name: Row 3 shows NULL name for customer_id 104. Missing names prevent proper identification and downstream joins; require fill or manual review. Recommendation: enforce NOT NULL, backfill from identity verification or set placeholder and flag for remediation.
- email: Row 2 (customer_id 103) has NULL email. Emails are often unique identifiers and required for contact; missing values break communications and uniqueness constraints. Recommendation: require email or alternate contact; backfill from source or flag for follow-up.
- city: Row 7 (customer_id 108) has NULL city. Affects segmentation/geolocation. Recommendation: impute from other sources, ask user, or use 'unknown' and flag.
- age: Row 11 (customer_id 111) has NULL age. If age is required for business rules, missing breaks analytics. Recommendation: impute cautiously (e.g., from DOB if available) or set NULL but mark for review.
- created_at: no explicit NULLs 